# FastAPI Production Engineering
### Async vs Sync | Cursor Pagination | Retry+Backoff | Dependency Injection | Validation

> **System throughout:** ShopFlow -- 500k daily users, FastAPI backend.
> Format: **Mental Model -> Real Scenario -> BEFORE -> AFTER -> Frameworks -> Nuances**

*Shift+Enter to run each cell*

## Setup

In [ ]:
from __future__ import annotations
import asyncio
import time
import random
from dataclasses import dataclass
from typing import Any
from collections.abc import Callable
print('Setup OK')

---
## 1 · Async vs Sync -- The Blocking Footgun

### Mental Model -- 'The Single Cashier'

```
WHAT   Python's asyncio is a single-threaded cooperative scheduler.
       One coroutine runs at a time; it voluntarily yields via 'await'.
WHY    If a coroutine blocks (sleeps, does sync I/O) without yielding,
       EVERY other request in the event loop is frozen until it finishes.
HOW    Use async-native I/O libraries (asyncpg, httpx, aiofiles).
       For unavoidable sync work, offload to a threadpool.
WHEN   Any FastAPI endpoint that does I/O must be truly async, or
       declare as plain 'def' so FastAPI runs it in a threadpool.
```

```
BROKEN -- 'async def' with sync sleep:
  Event loop: [req1 sleeping...30s freeze] -- req2,3,4 ALL WAIT

FIXED -- awaitable sleep:
  Event loop: [req1 await sleep -> yield] -> [req2 runs] -> [req3 runs]
              [req1 resumes when sleep done]
```

### Nuance 1: 'async def' that calls sync code is WORSE than 'def'
FastAPI runs plain `def` endpoints in a thread pool automatically -- they
never block the event loop. An `async def` endpoint that calls `time.sleep()`
blocks the ENTIRE event loop. The async keyword is a promise you must keep.

### Nuance 2: Not all async libraries are truly async
SQLAlchemy's standard session is sync. `requests` is sync. Many ORMs use
psycopg2 under the hood (sync). Check your driver: asyncpg and aiosqlite
are truly async; psycopg2 blocks. Using them in `async def` = footgun.

### Real-World Scenario -- ShopFlow PDF Invoice Generation

**Incident:** The `/invoice/{order_id}` endpoint was `async def` but called
a synchronous PDF library (`reportlab`) that takes 2 seconds per invoice.
During a billing run: 20 simultaneous requests -> event loop frozen for 40s.
All other endpoints (health checks, checkouts) timed out. Site appeared down.

**Fix 1 (quick):** Change `async def invoice()` to `def invoice()` -- FastAPI
runs it in a threadpool; event loop stays free.
**Fix 2 (proper):** Use `asyncio.get_event_loop().run_in_executor(None, fn)`
to explicitly offload to a threadpool from async context.

In [ ]:
# Simulate the event loop with a simple cooperative scheduler
# (In production this is FastAPI + uvicorn's actual asyncio loop)

completed_in_order: list[str] = []

# BEFORE -- 'async def' with a BLOCKING call (the footgun)
async def invoice_endpoint_BAD(order_id: str) -> str:
    # time.sleep blocks the event loop -- NO other coroutine runs
    time.sleep(0.05)   # simulates sync PDF generation
    completed_in_order.append(f'invoice-{order_id}')
    return f'invoice-{order_id}.pdf'


async def health_check() -> str:
    await asyncio.sleep(0)   # yield -- would run between invoice requests if loop free
    completed_in_order.append('health')
    return 'ok'


async def demo_bad():
    completed_in_order.clear()
    # Run 3 invoices + 1 health check concurrently
    await asyncio.gather(
        invoice_endpoint_BAD('A1'),
        invoice_endpoint_BAD('A2'),
        health_check(),
    )
    print('BAD order:', completed_in_order)
    print('  -> health check ran LAST despite yielding (was blocked by sync sleep)')


asyncio.run(demo_bad())

In [ ]:
# AFTER -- Option A: plain 'def' (FastAPI runs in threadpool automatically)
# Option B: async with run_in_executor for CPU/sync work

async def invoice_endpoint_GOOD(order_id: str) -> str:
    loop = asyncio.get_event_loop()
    # Offload sync work to threadpool -- event loop stays free
    def sync_pdf_generation():
        time.sleep(0.05)  # still takes time, but in a thread, not blocking loop
        return f'invoice-{order_id}.pdf'
    result = await loop.run_in_executor(None, sync_pdf_generation)
    completed_in_order.append(f'invoice-{order_id}')
    return result


async def demo_good():
    completed_in_order.clear()
    await asyncio.gather(
        invoice_endpoint_GOOD('B1'),
        invoice_endpoint_GOOD('B2'),
        health_check(),
    )
    print('GOOD order:', completed_in_order)
    print('  -> health check ran concurrently; event loop was never blocked')


asyncio.run(demo_good())

### Where This Is Seen in Real Frameworks

| Framework | Async nuance |
|-----------|_____________|
| **FastAPI** | Plain `def` endpoints run in a thread pool; `async def` runs in the event loop -- never mix |
| **SQLAlchemy async** | `AsyncSession` + `asyncpg` = truly async; `Session` + `psycopg2` = sync (use `run_sync`) |
| **httpx** | `AsyncClient` for async code; `Client` for sync -- never use `requests` in async def |
| **Redis** | `redis.asyncio.Redis` for async; `redis.Redis` is blocking |
| **Celery** | Task functions are sync; to call async code, use `asyncio.run()` inside the task |

---
## 2 · Cursor (Keyset) Pagination vs OFFSET

### Mental Model -- 'Bookmark vs Counting Pages'

```
WHAT   OFFSET skips rows by counting from the start every time.
       KEYSET remembers a cursor (last seen ID) and seeks to it via index.
WHY    OFFSET cost scales linearly with depth: page 9000 scans 180,000 rows.
       KEYSET cost is constant: always touches exactly 'limit' rows.
HOW    SQL: WHERE id > :cursor ORDER BY id LIMIT :n
       Return the last id as the 'next_cursor' for the client.
WHEN   Any list that can be large: orders, products, transactions, logs.
       OFFSET only acceptable for < 1000 rows or when jump-to-page is needed.
```

### Nuance 1: Keyset cannot jump to arbitrary pages
You can't do 'go to page 500' with keyset. If your UI needs page numbers,
either use OFFSET (accept the cost) or pre-compute page manifests.

### Nuance 2: Keyset cursor must be on an indexed, stable column
Using an unindexed column defeats the purpose -- the DB still does a full scan.
UUID as cursor: fine if indexed. Timestamp: risky if not unique (duplicates
within the same millisecond can cause skips).

### Nuance 3: Rows can appear/disappear between pages with OFFSET
If a row is inserted before page N's offset, the user sees a duplicate on page N+1.
Keyset is immune: 'WHERE id > 500' always starts exactly after id=500.

### Real-World Scenario -- ShopFlow Order History

**Incident:** The customer order history endpoint used OFFSET pagination.
A high-value customer had 50,000 orders. Scrolling to page 500 (offset=9980)
caused a 12-second DB query, timing out the request and spiking DB CPU to 100%.

**Fix:** Switch to keyset pagination using `ORDER BY id DESC WHERE id < :cursor`.
Page 500 now takes the same time as page 1: ~2ms.

In [ ]:

@dataclass(frozen=True)
class Row:
    id: int
    value: str


class FakeTable:
    def __init__(self, n: int):
        self._rows = [Row(i, f'order-{i}') for i in range(1, n+1)]
        self.scanned = 0

    def reset(self): self.scanned = 0

    # BEFORE -- OFFSET: scans from the beginning every time
    def page_offset(self, limit: int, offset: int) -> list[Row]:
        out = []
        for row in self._rows:
            self.scanned += 1
            if self.scanned <= offset: continue
            out.append(row)
            if len(out) == limit: break
        return out

    # AFTER -- KEYSET: binary seek to cursor (simulates B-tree index lookup)
    def page_keyset(self, limit: int, after_id: int = 0) -> list[Row]:
        lo, hi = 0, len(self._rows)
        while lo < hi:   # O(log n) seek -- the index does this in prod
            mid = (lo + hi) // 2
            if self._rows[mid].id <= after_id: lo = mid + 1
            else: hi = mid
        out = self._rows[lo:lo+limit]
        self.scanned += len(out)  # only rows returned, never more
        return out


table = FakeTable(200_000)
limit = 20

# Deep page (page 9000) via OFFSET
table.reset()
_ = table.page_offset(limit, offset=180_000)
offset_cost = table.scanned

# Same page via KEYSET
table.reset()
_ = table.page_keyset(limit, after_id=180_000)
keyset_cost = table.scanned

print(f'Deep page (offset=180,000, limit={limit})')
print(f'  OFFSET rows scanned: {offset_cost:,}  <- scales with depth')
print(f'  KEYSET rows scanned: {keyset_cost:,}  <- constant regardless of depth')
print(f'  Speedup: ~{offset_cost // keyset_cost}x')

# Show that early pages have the same KEYSET cost
table.reset()
_ = table.page_keyset(limit, after_id=0)  # page 1
print(f'Page 1 keyset cost: {table.scanned} rows -- same as page 9000')

In [ ]:
# FastAPI endpoint pattern for cursor pagination

from dataclasses import dataclass
import base64

@dataclass
class PageResponse:
    items:       list
    next_cursor: str | None  # None means no more pages
    page_size:   int


def encode_cursor(last_id: int) -> str:
    # Opaque cursor hides implementation detail (column name, type)
    return base64.b64encode(str(last_id).encode()).decode()

def decode_cursor(cursor: str) -> int:
    return int(base64.b64decode(cursor.encode()).decode())


def list_orders(after_cursor: str | None = None, limit: int = 20) -> PageResponse:
    after_id = decode_cursor(after_cursor) if after_cursor else 0
    # In real code: SELECT * FROM orders WHERE id > after_id ORDER BY id LIMIT limit+1
    all_ids = list(range(1, 201))  # simulated 200 orders
    page = [i for i in all_ids if i > after_id][:limit+1]

    has_more = len(page) > limit
    items    = page[:limit]
    cursor   = encode_cursor(items[-1]) if items and has_more else None
    return PageResponse(items=items, next_cursor=cursor, page_size=len(items))


p1 = list_orders()
print(f'Page 1: {p1.items[:3]}...  next_cursor: {p1.next_cursor}')

p2 = list_orders(after_cursor=p1.next_cursor)
print(f'Page 2: {p2.items[:3]}...  next_cursor: {p2.next_cursor}')

print('Cursor is opaque (base64-encoded id, not the raw id)')

### Where This Is Seen in Real Frameworks

| Framework | Cursor pagination |
|-----------|------------------|
| **SQLAlchemy** | `.where(Model.id > after_id).order_by(Model.id).limit(n)` |
| **Django ORM** | `.filter(id__gt=after_id).order_by('id')[:limit]` |
| **GitHub API** | `Link: <next_url>; rel='next'` header with opaque cursor |
| **Stripe API** | `starting_after=evt_xxx` parameter (keyset on event ID) |
| **Elasticsearch** | `search_after` parameter (keyset on sort value) |

---
## 3 · Retry + Exponential Backoff + Jitter

### Mental Model -- 'The Crowded Checkout'

```
WHAT   Retry failed requests with increasing delays + randomness to avoid
       all clients hitting the server at the same time after a blip.
WHY    Naive retry: 500 clients fail -> all retry at t=1s -> 500 hits
       -> server fails again -> retry at t=2s -> repeat ('retry storm').
HOW    Full jitter: wait = random(0, min(cap, base * 2^attempt))
       Desynchronizes clients across a time window.
WHEN   HTTP calls to external services (Stripe, SendGrid, S3).
       Internal microservice calls with transient failures.
```

### Nuance 1: NEVER retry on 4xx errors
4xx = client error (bad request, unauthorized, not found). Retrying won't
fix it -- you'll just spam the server with the same invalid request.
Only retry on: 429 (rate limit), 500/503 (server error), network timeouts.

### Nuance 2: Idempotency before retrying mutations
Retrying a POST /checkout without an idempotency key can cause double charges.
Always add an Idempotency-Key header to mutation endpoints before enabling retry.

### Nuance 3: Circuit breaker gates the retry
Retry handles transient blips (< 1s). Circuit breaker handles sustained outages.
Stack them: retry (3 attempts, 5s total) -> circuit breaker (opens after 5 fails).
Without the breaker, retry still amplifies load during a full outage.

In [ ]:

class TransientError(Exception): pass
class ClientError(Exception):    pass   # 4xx -- do NOT retry


# BEFORE -- naive retry: fixed delay, no jitter, retries 4xx
def call_api_BAD(fn, retries=3):
    for attempt in range(retries):
        try:
            return fn()
        except Exception:
            time.sleep(1)   # all clients retry at the same moment -> storm
    raise RuntimeError('failed')


# AFTER -- exponential backoff + full jitter + error discrimination
def call_api(
    fn: Callable,
    max_attempts: int = 4,
    base: float = 0.5,
    cap: float = 30.0,
    retryable: tuple = (TransientError,),
) -> Any:
    for attempt in range(max_attempts):
        try:
            return fn()
        except ClientError:
            raise   # 4xx: never retry, re-raise immediately
        except retryable:
            if attempt == max_attempts - 1: raise
            # Full jitter: random in [0, min(cap, base*2^attempt)]
            ceiling = min(cap, base * (2 ** attempt))
            wait = random.uniform(0, ceiling)
            print(f'  Attempt {attempt+1} failed. Retry in {wait:.2f}s')
            time.sleep(wait)


# Show jitter distribution (5 clients, attempt 2)
print('Jitter spread -- 5 clients, attempt 2 (base=0.5, cap=30):')
for i in range(5):
    ceiling = min(30.0, 0.5 * (2**2))
    w = random.uniform(0, ceiling)
    bar = '#' * int(w * 10)
    print(f'  client {i+1}: {w:.2f}s {bar}')
print('Spread avoids synchronized retry storm')

# Demonstrate: 4xx is NOT retried
_call_count = 0
def flaky_endpoint():
    global _call_count
    _call_count += 1
    raise ClientError('400 Bad Request -- bad payload')

try:
    call_api(flaky_endpoint)
except ClientError as e:
    print(f'Client error after {_call_count} call(s): {e} -- correct, not retried')

### Where This Is Seen in Real Frameworks

| Tool | Implementation |
|------|---------------|
| **tenacity** | `@retry(wait=wait_exponential(multiplier=0.5, max=30), retry=retry_if_exception_type(TransientError))` |
| **boto3** | Adaptive retry mode built-in; `max_attempts` in config |
| **httpx** | `transport=httpx.HTTPTransport(retries=3)` with custom retry logic |
| **Celery** | `@app.task(autoretry_for=(TransientError,), retry_backoff=True, max_retries=5)` |
| **Stripe SDK** | Built-in retry with idempotency key propagation |

---
## 4 · Dependency Injection -- FastAPI's Superpower

### Mental Model -- 'The Restaurant Supply Chain'

```
WHAT   Provide a component's dependencies from outside rather than
       having the component construct them itself.
WHY    Components that construct their own deps are untestable in isolation,
       unswappable, and implicitly coupled to infrastructure.
HOW    FastAPI's Depends() -- a function that returns a value, called per-request.
       Can be nested; FastAPI resolves the entire graph automatically.
WHEN   DB sessions, auth, config, HTTP clients, feature flags.
```

### Nuance 1: Depends() with yield is the scope mechanism
`async def get_db()` that `yield`s a session and closes it in the finally
block is FastAPI's request-scoped lifecycle. It's equivalent to a
context manager per request.

### Nuance 2: Dependencies are cached within a request
If two endpoints in the same request call `Depends(get_db)`, FastAPI returns
the SAME session object (default: `use_cache=True`). Two calls = one session.
This prevents N+1 sessions per request.

### Nuance 3: Override in tests without touching production code
`app.dependency_overrides[get_db] = lambda: FakeDB()` -- swap any dep
in tests with zero changes to endpoints. This is DIP made concrete.

In [ ]:
# Simulated FastAPI DI pattern (without the actual web framework)

class FakeDB:
    def __init__(self, name='prod'):
        self.name = name
        self.queries: list[str] = []
    def query(self, sql: str) -> list:
        self.queries.append(sql)
        return [{'id': 1, 'name': 'Widget'}]
    def close(self): pass


# BEFORE -- endpoint constructs its own database session
class ProductService_BAD:
    def get_products(self) -> list:
        db = FakeDB()   # constructed here -- untestable, unswappable
        return db.query('SELECT * FROM products')


# AFTER -- dependency injected
class ProductService:
    def __init__(self, db: FakeDB):
        self._db = db   # injected -- testable, swappable
    def get_products(self) -> list:
        return self._db.query('SELECT * FROM products')


# Production: inject real DB
prod_svc = ProductService(FakeDB('prod-db'))
print('Prod:', prod_svc.get_products())

# Test: inject fake DB -- zero changes to ProductService
fake_db = FakeDB('test-db')
test_svc = ProductService(fake_db)
result = test_svc.get_products()
print('Test:', result)
print('Queries recorded on fake:', fake_db.queries)
# In FastAPI: app.dependency_overrides[get_db] = lambda: FakeDB('test')
print('No real DB needed in tests!')

### Where This Is Seen in Real Frameworks

| Framework | DI usage |
|-----------|----------|
| **FastAPI** | `Depends(get_db)`, `Depends(get_current_user)`, `Depends(get_settings)` |
| **pytest** | Fixtures are DI: `def test_foo(db_session, client)` |
| **Django** | Not built-in; use `django-injector` or manual constructor injection |
| **SQLAlchemy** | `sessionmaker()` as a factory; inject the factory, not the session |
| **Celery** | Task dependencies injected via `bind=True`; `self.request` per-task context |

---
## 5 · Request Validation -- Defense at the Boundary

### Mental Model -- 'The Airport Security Checkpoint'

```
WHAT   Validate and parse all external input at the system boundary.
       Never trust data that crosses a trust boundary.
WHY    Bad data that enters the system causes bugs deep in business logic,
       making them hard to diagnose. Validate early, fail clearly.
HOW    Pydantic models as the single source of validation truth.
       Field-level constraints, custom validators, discriminated unions.
WHEN   Every HTTP request body, query parameter, path parameter,
       webhook payload, message queue message.
```

### Nuance 1: Validation != serialization != transformation
Pydantic does all three. `model_validator(mode='before')` runs before
field-level coercion. `model_validator(mode='after')` runs after.
Use 'before' for normalization (lowercase email), 'after' for cross-field rules
(end_date must be after start_date).

### Nuance 2: Never expose internal IDs in validation errors
A 422 error that says 'order 1234 not found' leaks internal state.
Generic errors at the boundary; specific errors inside the service.

### Nuance 3: Strict mode vs coercive mode
Default Pydantic: coerces '42' (string) to 42 (int). This is convenient
but hides bugs. Use `model_config = ConfigDict(strict=True)` for APIs
where you want exact types -- particularly for financial data.

In [ ]:
from dataclasses import dataclass

# Simulated validation (Pydantic-style without the dependency)

class ValidationError(ValueError): pass


class OrderRequest:
    def __init__(self, raw: dict):
        # Field-level validation
        if not isinstance(raw.get('email'), str) or '@' not in raw['email']:
            raise ValidationError('email must be a valid email address')
        if not isinstance(raw.get('quantity'), int) or raw['quantity'] < 1:
            raise ValidationError('quantity must be a positive integer')
        if not isinstance(raw.get('total'), (int, float)) or raw['total'] <= 0:
            raise ValidationError('total must be a positive number')
        # Normalize at boundary (never trust casing of email)
        self.email    = raw['email'].lower().strip()
        self.quantity = int(raw['quantity'])
        self.total    = float(raw['total'])
        # Cross-field validation
        if self.quantity > 999:
            raise ValidationError('quantity exceeds maximum order size (999)')

    def __repr__(self):
        return f'OrderRequest(email={self.email!r}, qty={self.quantity}, total={self.total})'


# Valid request
req = OrderRequest({'email': '  Alice@ShopFlow.COM  ', 'quantity': 3, 'total': 29.99})
print('Valid:', req)

# Invalid requests -- fail at the boundary with clear messages
for bad in [
    {'email': 'not-an-email', 'quantity': 3, 'total': 29.99},
    {'email': 'a@b.com', 'quantity': -1, 'total': 29.99},
    {'email': 'a@b.com', 'quantity': 1000, 'total': 29.99},
]:
    try:
        OrderRequest(bad)
    except ValidationError as e:
        print(f'Rejected at boundary: {e}')

### Where This Is Seen in Real Frameworks

| Framework | Validation usage |
|-----------|_________________|
| **FastAPI + Pydantic** | Every request body/query/path param is a Pydantic model; 422 returned automatically |
| **Django REST Framework** | `Serializer.is_valid()` -- validates and coerces; `validated_data` is safe to use |
| **Marshmallow** | `Schema.load()` -- validates and deserializes in one step |
| **Pydantic v2 strict mode** | `ConfigDict(strict=True)` -- no coercion, exact types required |
| **attrs + cattrs** | `cattrs.structure(raw, MyClass)` -- validates and converts |